In [16]:
from pathlib import Path

# Load corpus from chinese webnovels
corpus = ""
directory = Path("./novels-text")

for filepath in directory.iterdir():
    if filepath.is_file():
        with filepath.open('r', encoding='utf-8') as file:
            corpus += file.read()

print("Length of corpus:", len(corpus))

Length of corpus: 11142840


In [17]:
print("Sample text from corpus:")
print(corpus[:1000])

Sample text from corpus:
Coiling Dragon (盘龙) [Pan long]


By: I Eat Tomatoes （我吃西红柿）

Coiling Dragon (盘龙)

Panlong, aka Coiling Dragon, is a webnovel by popular Chinese Xianxia (fantasy/kung fu) writer I Eat Tomatoes （我吃西红柿）. This novel is current being translated by RWX. There are a total of 21 books spanning 800+ chapters, so sit back, buckle your seat belts, and get ready for one long ride!



INTRODUCTION: Empires rise and fall on the Yulan Continent. Saints, immortal beings of unimaginable power, battle using spells and swords, leaving swathes of destruction in their wake. Magical beasts rule the mountains, where the brave – or the foolish – go to test their strength. Even the mighty can fall, feasted on by those stronger. The strong live like royalty; the weak strive to survive another day.



This is the world which Linley is born into. Raised in the small town of Wushan, Linley is a scion of the Baruch clan, the clan of the once-legendary Dragonblood Warriors. Their fame once s

In [18]:
# Unique chars and how many
chars = sorted(set(corpus))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !#%'()*+,-./0123456789:;<=?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxyz~çéï–—‘’“”…吃圣我斗柿气猛盘红西迅龙（），😉
111


In [19]:
ctoi = {c:i for i,c in enumerate(chars)}
itoc = {i:c for i,c in enumerate(chars)}
def encode(string: str) -> int:
    return [ctoi[c] for c in string]
def decode(indices: list) -> str:
    return ''.join([itoc[i] for i in indices])

print(encode("hello child"))
print(decode(encode("brug sandwich")))

[65, 62, 69, 69, 72, 1, 60, 65, 66, 69, 61]
brug sandwich


In [20]:
import torch
data = torch.tensor(encode(corpus), dtype=torch.long)#
print(data.shape, data.dtype)
print(data[:100])

torch.Size([11142840]) torch.int64
tensor([ 31,  72,  66,  69,  66,  71,  64,   1,  32,  75,  58,  64,  72,  71,
          1,   6, 102, 106,   7,   1,  55,  44,  58,  71,   1,  69,  72,  71,
         64,  56,   0,   0,   0,  30,  82,  24,   1,  37,   1,  33,  58,  77,
          1,  48,  72,  70,  58,  77,  72,  62,  76,   1, 107,  97,  95, 104,
        103,  99, 108,   0,   0,  31,  72,  66,  69,  66,  71,  64,   1,  32,
         75,  58,  64,  72,  71,   1,   6, 102, 106,   7,   0,   0,  44,  58,
         71,  69,  72,  71,  64,  10,   1,  58,  68,  58,   1,  31,  72,  66,
         69,  66])


In [21]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [36]:
block_size = 8
chunky = train_data[:block_size+1]
print(decode(chunky[:-1].tolist()))

Coiling 


In [37]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"When {context} then you get {target}")

When tensor([31]) then you get 72
When tensor([31, 72]) then you get 66
When tensor([31, 72, 66]) then you get 69
When tensor([31, 72, 66, 69]) then you get 66
When tensor([31, 72, 66, 69, 66]) then you get 71
When tensor([31, 72, 66, 69, 66, 71]) then you get 64
When tensor([31, 72, 66, 69, 66, 71, 64]) then you get 1
When tensor([31, 72, 66, 69, 66, 71, 64,  1]) then you get 32


In [41]:
batch_size = 10
block_size = 8

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print("Batch shape:")
print(xb.shape)
print(xb)
print("Batch target shape:")
print(yb.shape)
print(yb)

Batch shape:
torch.Size([10, 8])
tensor([[77,  1, 80, 66, 76, 65,  1, 77],
        [61,  1, 70, 72, 76, 77,  1, 69],
        [ 1, 76, 66, 70, 73, 69, 82,  1],
        [63,  2,  0,  0, 92, 53, 72, 78],
        [72, 79, 62, 75, 62, 61,  1, 65],
        [ 1, 77, 65, 62,  1, 76, 65, 72],
        [92, 51, 65, 82, 91, 61,  1, 82],
        [ 0, 92, 47, 70, 58, 76, 65,  2],
        [78, 69, 61,  1, 76, 77, 72, 73],
        [71, 61,  1, 40, 66, 71, 69, 62]])
Batch target shape:
torch.Size([10, 8])
tensor([[ 1, 80, 66, 76, 65,  1, 77, 72],
        [ 1, 70, 72, 76, 77,  1, 69, 66],
        [76, 66, 70, 73, 69, 82,  1, 77],
        [ 2,  0,  0, 92, 53, 72, 78,  1],
        [79, 62, 75, 62, 61,  1, 65, 66],
        [77, 65, 62,  1, 76, 65, 72, 78],
        [51, 65, 82, 91, 61,  1, 82, 72],
        [92, 47, 70, 58, 76, 65,  2, 93],
        [69, 61,  1, 76, 77, 72, 73,  1],
        [61,  1, 40, 66, 71, 69, 62, 82]])


In [46]:
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        if targets is None:
            # if we are not given targets, we are in inference mode
            loss = None
        else:
        # idx and targets are both (batch_size, block_size) tensor
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # reshape to (B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx, None)
            # focus only on the last time step
            logits = logits[:, -1, :]
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
    
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))

torch.Size([80, 111])
tensor(5.2449, grad_fn=<NllLossBackward0>)

迅😉😉#=2BiEMt柿红迅i）lP/C(]V+P…，PGB'euJ猛O!s[猛9;气wJ.:e;，[ï柿5e
3.7)H.]“,–:Y”y西I[tI红/西~h3ExG）oUQHf_Y.N]i+[吃)


In [48]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [50]:
batch_size = 32
for steps in range(100000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True) # faster
    loss.backward()
    optimizer.step()

print(loss.item())

2.4832098484039307


In [55]:
print(decode(m.generate(idx, max_new_tokens=500)[0].tolist()))



Dal s hed throe Unst. mofinthasey, wata They tassthas, th.” rbu he isse w wer y osey tod tealeastan Li nesfaghederitig.

Howe t 13, s qut os bspevimain ph-ldle bago ont walou woushasherongyeang Sonderurow Hedingoncedeery wing aliatis a’trouisewhasang; y’s are ve lela venerouerikly stacle wends, atasthe hesucee Af antheerosois clintheche? ‘od eng t bothers berdnoe st su, f-lld, fldr, as?” But!” t.
Lif od dvethintoloucugexth tindneyd.”
“Beeramowne Che wh Pringothgerical Dy. f wnctand fl Burortt f


# Getting to Self Attention
math trick.

In [57]:
B,T,C = 4,8,2
x = torch.randn(B,T,C)
print(x.shape)

torch.Size([4, 8, 2])


In [67]:
xbow = torch.zeros(B,T,C) # bag of words
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1,:] # (t, C)
        xbow[b,t] = torch.mean(xprev, 0) # (C,)

In [71]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)


True

In [72]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)


True

In [ ]:
# version 4: self-attention!
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

In [73]:
x[0], xbow[0]

(tensor([[ 1.2060,  0.6762],
         [-0.8010, -1.4262],
         [ 0.8237, -1.6908],
         [-0.0900,  0.0404],
         [ 0.6898,  0.2881],
         [ 0.9512,  0.7304],
         [ 0.1258,  1.0849],
         [ 1.5477,  0.3217]]),
 tensor([[ 1.2060,  0.6762],
         [ 0.2025, -0.3750],
         [ 0.4096, -0.8136],
         [ 0.2847, -0.6001],
         [ 0.3657, -0.4225],
         [ 0.4633, -0.2303],
         [ 0.4151, -0.0424],
         [ 0.5567,  0.0031]]))

In [74]:
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, dim=1, keepdim=True)
b = torch.randint(0,10, (3,2)).float()
c = a @ b
print('a')
print(a)
print('b')
print(b)
print('c')
print(c)

a
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b
tensor([[7., 7.],
        [1., 5.],
        [3., 3.]])
c
tensor([[7.0000, 7.0000],
        [4.0000, 6.0000],
        [3.6667, 5.0000]])
